# Post-hoc set 3 — VLM transcription of the 500 pages (H3) on ViDoRe DocVQA (Colab)

Thin wrapper around `scripts/run_posthoc3_gpu.py --stage ocr` (Qwen3.5-4B with vLLM; pre-registered in `e70f2a7`).
Use a **fresh runtime**: vLLM installs its own torch. If Qwen3.5 does not run within one attempt, the pre-registered
fallback is `--fallback` (Qwen3-VL-4B-Instruct); record the failed attempt.

**Before running:** `Runtime → Change runtime type → L4 GPU`.

In [ ]:
!nvidia-smi

In [ ]:
# Idempotent: clones on first run, pulls the latest commit when the runtime is reused.
import os
if not os.path.isdir("/content/colpali-hybrid-retrieval"):
    !git clone https://github.com/ismailucrn/colpali-hybrid-retrieval.git /content/colpali-hybrid-retrieval
%cd /content/colpali-hybrid-retrieval
!git pull -q
!git log -1 --format='%H %s'

In [ ]:
!pip install -q vllm==0.33.0 datasets==4.8.5 pyyaml

In [ ]:
# Smoke test (never reported).
!python scripts/run_posthoc3_gpu.py --stage ocr --limit 4
!python -c "import json;d=json.load(open('results/vlm_ocr_docvqa_smoke/page_texts.json'));[print(k, repr(v[:300])) for k,v in d.items()]"

In [ ]:
!python scripts/run_posthoc3_gpu.py --stage ocr

In [ ]:
!pip freeze > results/vlm_ocr_docvqa/pip_freeze_colab.txt
!ls -la results/vlm_ocr_docvqa
!cat results/vlm_ocr_docvqa/metrics.json

In [ ]:
# Zip the outputs and download them (binary files cannot be copied from the page).
# Guard: "Run all" continues after a failed shell command, so only download a complete run.
import os
for d in ['results/vlm_ocr_docvqa']:
    assert os.path.exists(d + "/metrics.json"), d + " did not finish; nothing to download"
!zip -q -r /content/posthoc3_vlm_ocr.zip results/vlm_ocr_docvqa
!sha256sum /content/posthoc3_vlm_ocr.zip
from google.colab import files
files.download("/content/posthoc3_vlm_ocr.zip")